# Chain Fine-Tune: RDD2022 → N-RDD2024

**Experiment:** COCO → RDD2022+Pothole600 → N-RDD2024 (10 classes)  
**Hypothesis:** Starting from `rtdetr_l_rdd2022.pt` (mAP50=0.465) gives the N-RDD2024
fine-tune a better initialisation — the backbone already knows what road damage looks like.

| Checkpoint | Init | Dataset | mAP50 val |
|---|---|---|---|
| `rtdetr_l_rdd2022.pt` | COCO → RDD2022+P600 | RDD2022+P600 | 0.465 |
| `rtdetr_l_nrdd2024.pt` | COCO → N-RDD2024 | N-RDD2024 | 0.577 |
| `rtdetr_l_rdd2022_nrdd2024.pt` | COCO → RDD2022 → N-RDD2024 | N-RDD2024 | **this run** |

**Mosaic note (from previous N-RDD2024 analysis):**  
In the COCO→N-RDD2024 run, `best.pt` was saved at epoch 47 — which coincides exactly
with the Ultralytics mosaic cutoff (mosaic disabled in the final ~10 epochs of training).
This is structural Ultralytics behaviour, not a bug. `mosaic=0.860` (PSO-found) already
partially mitigates low-contrast frames by randomly skipping mosaic 14% of the time.
The same setting is preserved here for a controlled comparison.

**References:**  
RT-DETR (Zhao et al., 2024) · N-RDD2024 (Kaya & Çodur, 2024) · PSO (Kennedy & Eberhart, 1995)
· Label smoothing (Szegedy et al., 2016) · Albumentations (Buslaev et al., 2020)

## Cell 1 — Install

In [ ]:
import subprocess, sys

# Pin ultralytics to 8.3.143 — exact version used in the working notebook
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'ultralytics==8.3.143', 'pyyaml', 'tqdm', 'psutil', 'py-cpuinfo'
], check=True)

import torch, ultralytics, PIL
print(f'PyTorch     : {torch.__version__}')
print(f'Ultralytics : {ultralytics.__version__}')
print(f'Pillow      : {PIL.__version__}')
print('Install complete.')


## Cell 2 — Imports & logger

In [ ]:
import os, torch, ultralytics, PIL, logging, json, shutil, time
import yaml, sys, threading
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm

# Patch check_amp: skips the yolo26n.pt download probe
# This is what the working notebook does — without it Ultralytics
# downloads yolo26n.pt for AMP verification before every training run
import ultralytics.utils.checks as _uc
_uc.check_amp = lambda model: True

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(message)s',
    datefmt='%H:%M:%S',
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)

if not torch.cuda.is_available():
    raise RuntimeError('No GPU — Kaggle → Settings → Accelerator → GPU T4 x2')

torch.backends.cudnn.benchmark     = True
torch.backends.cudnn.deterministic = False

logger.info('PyTorch     : %s', torch.__version__)
logger.info('Ultralytics : %s', ultralytics.__version__)
logger.info('Pillow      : %s', PIL.__version__)
for i in range(torch.cuda.device_count()):
    logger.info('  GPU %d: %s  %.1f GB', i,
                torch.cuda.get_device_name(i),
                torch.cuda.get_device_properties(i).total_memory / 1e9)


## Cell 3 — Paths

Confirmed folder structure (from `evaluate_nrdd2024_final.ipynb`):
```
N-RDD2024Road damage and defects/
  Training and Validation Dataset/
    Czech_txt/train/images/   Czech_txt/train/labels/
    Czech_txt/valid/images/   Czech_txt/valid/labels/
    India_txt/...
    Japan_txt/...
    Norway_txt/...
    United_States_txt/...
    China_MotorBike_txt/...
  Test Dataset/
    Czech/Czech/test/images/
    ...
```

In [ ]:
# ── Confirmed paths (from evaluate_nrdd2024_final.ipynb + user confirmation) ──
NRDD_BASE = Path(
    '/kaggle/input/datasets/sannyshankaranml/n-rdd2024'
    '/N-RDD2024Road damage and defects'
)
TRAIN_BASE = NRDD_BASE / 'Training and Validation Dataset'

# RDD2022 PSO checkpoint — THE chain init (key difference from COCO→NRDD run)
RDD2022_CKPT = Path(
    '/kaggle/input/datasets/paraschiv/rdd2022-fine-tuned-rt-detr/best.pt'
)

# Phase 1 checkpoint — last.pt from Phase 1 uploaded to fine-tune-phase1 dataset
# This is what Phase 2 loads. Separate from RDD2022_CKPT which only initialises Phase 1.
PHASE1_CKPT = Path(
    '/kaggle/input/datasets/paraschiv/chain-fine-tune-phase1/last.pt'
)

# Working dirs
WORK        = Path('/kaggle/working')
DATASET_DIR = WORK / 'dataset'
SAVE_DIR    = WORK / 'chain_finetune_nrdd2024'
for d in [DATASET_DIR, SAVE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Hard assertions — fail immediately if paths are wrong
assert NRDD_BASE.exists(),    f'NRDD_BASE not found: {NRDD_BASE}'
assert TRAIN_BASE.exists(),   f'TRAIN_BASE not found: {TRAIN_BASE}'
assert RDD2022_CKPT.exists(), f'RDD2022 checkpoint not found: {RDD2022_CKPT}'
# PHASE1_CKPT asserted in Phase 2 cell — it doesn't exist yet when Cell 3 runs
# (you need to upload last.pt to fine-tune-phase1 dataset first)

logger.info('NRDD_BASE    : %s', NRDD_BASE)
logger.info('TRAIN_BASE   : %s', TRAIN_BASE)
logger.info('PHASE1_CKPT  : %s', PHASE1_CKPT)
logger.info('RDD2022_CKPT : %s  (%.1f MB)', RDD2022_CKPT,
            RDD2022_CKPT.stat().st_size / 1e6)

# Show country folders and image counts — confirms structure before copying
logger.info('Country folders in TRAIN_BASE:')
for cf in sorted(TRAIN_BASE.iterdir()):
    if not cf.is_dir():
        continue
    n_train = len(list((cf / 'train' / 'images').glob('*.jpg'))) \
              if (cf / 'train' / 'images').exists() else 0
    n_valid = len(list((cf / 'valid' / 'images').glob('*.jpg'))) \
              if (cf / 'valid' / 'images').exists() else 0
    logger.info('  %-26s  train=%4d  valid=%4d', cf.name, n_train, n_valid)

## Cell 4 — Build dataset

**No class ID remapping needed.** The `_txt` Roboflow export already uses canonical 0–9 IDs.

**Imbalance mitigations (train split only):**
- Image-level oversampling: D90 ×10, D30 ×4, D80 ×2, D50 ×2
- `cls=0.8` (raised from PSO default 0.487)
- `label_smoothing=0.1` (Szegedy et al., 2016)
- Focal Loss γ=2.0 is built into RT-DETR — not user-configurable

**Mosaic:** `mosaic=0.860` (PSO-found). Ultralytics disables mosaic in the final
epochs automatically — this is what caused best.pt to be saved at epoch 47 in the
previous COCO→N-RDD2024 run (the model saw cleaner crops and precision improved).
Keeping mosaic=0.860 here for controlled comparison.

In [ ]:
# Canonical 10-class N-RDD2024 taxonomy
# Ref: Kaya & Codur, Mendeley Data V3, doi:10.17632/27c8pwsd6v.3
CANONICAL_CLASSES = [
    'longitudinal_crack',       # D00  ID 0
    'transverse_crack',         # D10  ID 1
    'alligator_crack',          # D20  ID 2
    'repaired_crack',           # D30  ID 3  — minority (311 instances)
    'pothole',                  # D40  ID 4
    'pedestrian_crossing_blur', # D50  ID 5  — addresses FP on crossings
    'lane_line_blur',           # D60  ID 6  — addresses FP on lane markings
    'manhole_cover',            # D70  ID 7
    'patchy_road',              # D80  ID 8  — minority
    'rutting',                  # D90  ID 9  — minority (41 instances)
]

# Oversampling factors for minority classes (train split only)
# 442:1 ratio between D00 (18163) and D90 (41) — explicit mitigation required
OVERSAMPLE = {9: 10, 3: 4, 8: 2, 5: 2}  # canonical class_id: n_copies

# Create output split dirs
for split in ['train', 'valid']:
    (DATASET_DIR / split / 'images').mkdir(parents=True, exist_ok=True)
    (DATASET_DIR / split / 'labels').mkdir(parents=True, exist_ok=True)

stats         = defaultdict(int)
country_stats = {}

for country_dir in sorted(TRAIN_BASE.iterdir()):
    if not country_dir.is_dir():
        continue

    cname = country_dir.name   # e.g. 'Czech_txt'
    country_stats[cname] = defaultdict(int)

    for src_split, dst_split in [('train', 'train'), ('valid', 'valid')]:
        img_dir = country_dir / src_split / 'images'
        lbl_dir = country_dir / src_split / 'labels'

        if not img_dir.exists():
            continue

        images = sorted(img_dir.glob('*.jpg')) + sorted(img_dir.glob('*.png'))

        for img_path in images:
            lbl_path = lbl_dir / (img_path.stem + '.txt')
            if not lbl_path.exists():
                continue

            # Determine oversampling factor from label contents (train only)
            copies = 1
            if dst_split == 'train':
                class_ids = set()
                for line in lbl_path.read_text().strip().split('\n'):
                    if line.strip():
                        class_ids.add(int(line.split()[0]))
                if class_ids:
                    copies = max(OVERSAMPLE.get(cid, 1) for cid in class_ids)

            for copy_idx in range(copies):
                suffix  = f'_{copy_idx}' if copy_idx > 0 else ''
                stem    = f'{cname}_{img_path.stem}{suffix}'
                dst_img = DATASET_DIR / dst_split / 'images' / (stem + img_path.suffix)
                dst_lbl = DATASET_DIR / dst_split / 'labels' / (stem + '.txt')

                # shutil.copy2 — NOT symlinks (symlinks fail under DDP)
                if not dst_img.exists():
                    shutil.copy2(img_path, dst_img)
                if not dst_lbl.exists():
                    shutil.copy2(lbl_path, dst_lbl)  # no remap needed

                stats[dst_split] += 1
                country_stats[cname][dst_split] += 1

    logger.info('  %-28s  train=%d  valid=%d',
                cname,
                country_stats[cname]['train'],
                country_stats[cname]['valid'])

logger.info('=' * 55)
logger.info('TOTAL  train=%d  valid=%d', stats['train'], stats['valid'])

# Hard stop — if 0 images were copied DDP will crash with a confusing error
assert stats['train'] > 0, (
    f'FATAL: 0 training images assembled. '
    f'Check TRAIN_BASE: {TRAIN_BASE}'
)
assert stats['valid'] > 0, 'FATAL: 0 validation images assembled.'

# Spot-check: show first 3 physical files in train/images
sample = sorted((DATASET_DIR / 'train' / 'images').iterdir())[:3]
logger.info('First 3 train images (confirming real files, not symlinks):')
for p in sample:
    logger.info('  %-50s  %.0f KB', p.name, p.stat().st_size / 1024)

# Write dataset.yaml
yaml_path   = DATASET_DIR / 'dataset.yaml'
dataset_cfg = {
    'path':  str(DATASET_DIR),
    'train': 'train/images',
    'val':   'valid/images',
    'nc':    10,
    'names': CANONICAL_CLASSES,
}
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_cfg, f, default_flow_style=False)
logger.info('dataset.yaml written -> %s', yaml_path)

## Cell 5 — Persistence watcher

In [ ]:
_watcher_active = True

def _sync():
    """Background thread: sync training outputs to SAVE_DIR every 60s."""
    while _watcher_active:
        time.sleep(60)
        run_dir = Path('/kaggle/working/runs/detect/chain_nrdd2024')
        if run_dir.exists():
            for src in run_dir.rglob('*'):
                if src.is_file() and src.suffix in ('.pt', '.csv', '.xlsx', '.json', '.png'):
                    try:
                        shutil.copy2(src, SAVE_DIR / src.name)
                    except Exception:
                        pass

threading.Thread(target=_sync, daemon=True).start()
logger.info('Persistence watcher active — syncing to %s every 60s', SAVE_DIR)

## Cell 6 — PSO hyperparameters

In [ ]:
# Try to load actual PSO file from Kaggle input
PSO_FILE = Path(
    '/kaggle/input/datasets/paraschiv/rdd2022-fine-tuned-rt-detr/pso_best.json'
)

if PSO_FILE.exists():
    with open(PSO_FILE) as f:
        pso = json.load(f)
    logger.info('PSO params loaded from: %s', PSO_FILE)
else:
    # Actual values from the RDD2022 PSO search
    # Ref: Kennedy & Eberhart (1995); Young et al. (2015)
    # Working with actual search results — not mock data
    pso = {
        'lr0':           4.47e-4,
        'weight_decay':  5.27e-4,
        'warmup_epochs': 1,
        'mosaic':        0.860,   # PSO-found; also mitigates low-contrast mosaic frames
        'mixup':         0.205,   # Ref: Zhang et al. (2018)
        'box':           7.685,
        'cls':           0.487,   # base; overridden below for N-RDD2024 imbalance
    }
    logger.warning('pso_best.json not found — using hardcoded PSO values from RDD2022 run')

# N-RDD2024 cls override — 442:1 imbalance between D00 and D90
CLS_WEIGHT = 0.8

logger.info('PSO params   : %s', pso)
logger.info('cls override : %.1f (was %.3f, raised for 442:1 imbalance)',
            CLS_WEIGHT, pso['cls'])
logger.info('mosaic       : %.3f — PSO-found; Ultralytics disables mosaic'
            ' in final epochs automatically (caused best.pt at ep47 in'
            ' previous COCO→N-RDD2024 run)', pso['mosaic'])

## Cell 8 — Phase 2: Full fine-tune (70 epochs)

All layers unfrozen. LR reduced 10× to prevent catastrophic forgetting of RDD2022 features.
This is especially important for the chain fine-tune: the backbone carries more
task-relevant information than a COCO init, so a lower LR preserves it better.

70 epochs gives more headroom than the previous COCO→N-RDD2024 run (57 epochs,
best at ep47). Early stopping (`patience=20`) terminates if the model plateaus.

In [ ]:
# ── Phase 1 checkpoint — loaded from fine-tune-phase1 Kaggle dataset ──────────
# Upload the last.pt from Phase 1 to your fine-tune-phase1 dataset before running.
# last.pt == best.pt here because mAP50 was still rising at epoch 10 (last epoch).
assert PHASE1_CKPT.exists(), (
    f'Phase 1 checkpoint not found: {PHASE1_CKPT}\n'
    'Upload runs/detect/.../weights/last.pt to the fine-tune-phase1 Kaggle dataset.'
)
logger.info('Phase 1 checkpoint : %s  (%.1f MB)', PHASE1_CKPT,
            PHASE1_CKPT.stat().st_size / 1e6)

logger.info('Phase 2 — all layers unfrozen, 70 epochs, patience=20')
logger.info('LR: %.2e  (10x lower than Phase 1 lr0=%.2e)',
            pso['lr0'] * 0.1, pso['lr0'])

# Fresh model load — do NOT use resume=True
# resume=True would inherit Phase 1 optimizer state and produce wrong LR
# Ref: is_resuming_phase2 pattern from train.py
model_p2 = RTDETR(str(PHASE1_CKPT))

t0 = time.time()
model_p2.train(
    data            = str(yaml_path),
    epochs          = 70,
    imgsz           = 640,
    batch           = BATCH,
    optimizer       = 'AdamW',
    lr0             = pso['lr0'] * 0.1,  # 10x lower for full fine-tune
    weight_decay    = pso['weight_decay'],
    warmup_epochs   = 1,
    mosaic          = pso['mosaic'],
    mixup           = pso['mixup'],
    box             = pso['box'],
    cls             = CLS_WEIGHT,
    label_smoothing = 0.1,
    freeze          = 0,                 # all layers unfrozen
    amp             = True,
    device          = DEVICE,
    workers         = 4,
    patience        = 20,
    exist_ok        = True,
    project         = '/kaggle/working/runs/detect',  # absolute — avoids double-nesting
    name            = 'chain_nrdd2024_phase2',  # different name to avoid path collision
    val             = True,
    plots           = True,
)
logger.info('Phase 2 complete in %.1f min', (time.time() - t0) / 60)

_watcher_active = False

p2_best = Path('/kaggle/working/runs/detect/chain_nrdd2024_phase2/weights/best.pt')
if p2_best.exists():
    out = SAVE_DIR / 'rtdetr_l_rdd2022_nrdd2024.pt'
    shutil.copy2(p2_best, out)
    logger.info('Final checkpoint -> %s  (%.1f MB)', out,
                out.stat().st_size / 1e6)
else:
    logger.error('best.pt not found at %s — check run folder', p2_best)
    # Fallback: save last.pt
    p2_last = Path('/kaggle/working/runs/detect/chain_nrdd2024_phase2/weights/last.pt')
    if p2_last.exists():
        out = SAVE_DIR / 'rtdetr_l_rdd2022_nrdd2024_last.pt'
        shutil.copy2(p2_last, out)
        logger.warning('Saved last.pt instead -> %s', out)


## Cell 9 — Results & comparison

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

results_csv = Path('/kaggle/working/runs/detect/chain_nrdd2024_phase2/results.csv')
if not results_csv.exists():
    results_csv = SAVE_DIR / 'results.csv'  # fall back to saved copy

if not results_csv.exists():
    logger.error('results.csv not found — training may not have completed')
else:
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    shutil.copy2(results_csv, SAVE_DIR / 'results.csv')

    bi      = df['metrics/mAP50(B)'].idxmax()
    map50   = df['metrics/mAP50(B)'].max()
    map9595 = df['metrics/mAP50-95(B)'].iloc[bi]
    prec    = df['metrics/precision(B)'].iloc[bi]
    rec     = df['metrics/recall(B)'].iloc[bi]

    logger.info('=' * 58)
    logger.info('CHAIN FINE-TUNE FINAL RESULTS')
    logger.info('  Best epoch    : %d / %d', bi + 1, len(df))
    logger.info('  mAP50         : %.4f', map50)
    logger.info('  mAP50-95      : %.4f', map9595)
    logger.info('  Precision     : %.4f', prec)
    logger.info('  Recall        : %.4f', rec)
    logger.info('--- three-way comparison ---')
    logger.info('  COCO → RDD2022                  mAP50=0.465')
    logger.info('  COCO → N-RDD2024                mAP50=0.577  (best ep47/57)')
    logger.info('  COCO → RDD2022 → N-RDD2024      mAP50=%.3f  (this run, best ep%d/%d)',
                map50, bi + 1, len(df))
    logger.info('=' * 58)

    # Mosaic cutoff note
    logger.info('Mosaic note: Ultralytics disables mosaic automatically in'
                ' the final ~10 epochs. The resulting drop in augmentation'
                ' diversity often moves best.pt to that window (as seen at'
                ' ep47 in the previous run). Check if best epoch is in the'
                ' last 10 epochs of this run.')
    if bi + 1 > len(df) - 12:
        logger.info('  -> best.pt IS in the mosaic-off window (ep%d, last %d epochs)',
                    bi + 1, len(df) - bi)

    epochs = range(1, len(df) + 1)
    fig, axes = plt.subplots(2, 2, figsize=(14, 9), facecolor='#f8f7f4')
    fig.suptitle(
        'Chain Fine-Tune: COCO → RDD2022 → N-RDD2024\n'
        f'Best epoch {bi+1}/{len(df)}  mAP50={map50:.4f}  mAP50-95={map9595:.4f}',
        fontsize=12, fontweight='bold'
    )

    def _style(ax, title):
        ax.set_facecolor('#f0ede8')
        ax.set_title(title, fontsize=10, fontweight='600')
        ax.axvline(bi + 1, color='#aaa', ls='--', lw=0.8,
                   label=f'best.pt ep{bi+1}')
        ax.grid(axis='y', color='#ddd', lw=0.5, alpha=0.7)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.set_xlabel('Epoch', fontsize=8)

    axes[0,0].plot(epochs, df['metrics/mAP50(B)'],    color='#1D9E75', lw=1.8, label='mAP50')
    axes[0,0].plot(epochs, df['metrics/mAP50-95(B)'], color='#7F77DD', lw=1.8, label='mAP50-95')
    # Reference lines from previous runs
    axes[0,0].axhline(0.577, color='#1D9E75', lw=0.8, ls=':', alpha=0.6, label='COCO→NRDD 0.577')
    axes[0,0].axhline(0.465, color='#aaa',    lw=0.8, ls=':', alpha=0.6, label='RDD2022   0.465')
    _style(axes[0,0], 'mAP (dotted = previous runs)')
    axes[0,0].legend(fontsize=7)

    axes[0,1].plot(epochs, df['metrics/precision(B)'], color='#D85A30', lw=1.8, label='Precision')
    axes[0,1].plot(epochs, df['metrics/recall(B)'],    color='#BA7517', lw=1.8, label='Recall')
    _style(axes[0,1], 'Precision & Recall')
    axes[0,1].legend(fontsize=8)

    axes[1,0].plot(epochs, df['train/giou_loss'], color='#1D9E75', lw=1.6, label='Train GIoU')
    axes[1,0].plot(epochs, df['train/cls_loss'],  color='#7F77DD', lw=1.6, label='Train Cls')
    _style(axes[1,0], 'Train Losses')
    axes[1,0].legend(fontsize=8)

    axes[1,1].plot(epochs, df['train/giou_loss'], color='#1D9E75', lw=1.6, ls='-',  label='Train GIoU')
    axes[1,1].plot(epochs, df['val/giou_loss'],   color='#1D9E75', lw=1.6, ls='--', label='Val GIoU')
    _style(axes[1,1], 'Train vs Val GIoU (overfitting check)')
    axes[1,1].legend(fontsize=8)

    plt.tight_layout()
    plot_path = SAVE_DIR / 'chain_results.png'
    plt.savefig(plot_path, dpi=140, bbox_inches='tight')
    plt.show()
    logger.info('Plot saved -> %s', plot_path)

# Final file inventory
logger.info('Outputs in %s:', SAVE_DIR)
for f in sorted(SAVE_DIR.iterdir()):
    logger.info('  %-48s  %.1f MB', f.name, f.stat().st_size / 1e6)